# Deploy Decision 2.0 Kai 0.6B on a CPU endpoint on SageMaker AI

Deploy [Decision 2.0 Kai 0.6B](https://huggingface.co/vllm-sr/Decision-2.0-Kai-0.6B)
on one `ml.c7i.2xlarge` real-time endpoint with no GPU. Invoke its native Choice,
Yes/No (`noul`), and Score outputs through Transformers `AutoModel` and
`system_one`. The checkpoint and its remote runtime code are pinned to the same
revision. Set `DECISION_MODEL=eos` to deploy
[Decision 2.0 Eos 0.8B](https://huggingface.co/vllm-sr/Decision-2.0-Eos-0.8B)
(16,384-token context) through the same cells.

It runs on the maintained AWS PyTorch 2.14 **CPU** SageMaker DLC, through the small
serving image that this notebook builds with AWS CodeBuild. Read
[README.md](README.md) for prerequisites, licensing, and runtime details. This is
an evaluation example.

Run from this directory. Run the cleanup cell when finished, including after a
failed deployment once SageMaker allows endpoint deletion.

## 1. Environment preparation

In [ ]:
%pip install --upgrade 'boto3>=1.43.1,<2' --quiet

In [ ]:
import json
import math
import os
import statistics
import tarfile
import time
import uuid
from pathlib import Path

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

session = boto3.Session()
region = session.region_name or "us-east-1"
sm = session.client("sagemaker", region_name=region)
runtime = session.client(
    "sagemaker-runtime", region_name=region,
    config=Config(read_timeout=70, retries={"max_attempts": 0}),
)
s3 = session.client("s3", region_name=region)
iam = session.client("iam")
logs = session.client("logs", region_name=region)
identity = session.client("sts").get_caller_identity()
account_id = identity["Account"]

# Both models share the decision2 runtime (qwen-full profile) and this adapter.
MODELS = {
    "kai": {
        "model_id": "vllm-sr/Decision-2.0-Kai-0.6B",
        "revision": "cc7f30cda9e7c7b28c5f71ef33f01e09385f1c4b",
        "model_name": "Decision-2.0-Kai-0.6B",
        "loaded_parameters": 597103104,
        "max_input_tokens": 8192,
    },
    "eos": {
        "model_id": "vllm-sr/Decision-2.0-Eos-0.8B",
        "revision": "52cad37ef21f598302fe964cbfa164739cb965d5",
        "model_name": "Decision-2.0-Eos-0.8B",
        "loaded_parameters": 753446208,
        "max_input_tokens": 16384,
    },
}
model_choice = os.environ.get("DECISION_MODEL", "kai").lower()
if model_choice not in MODELS:
    raise ValueError(f"DECISION_MODEL must be one of {sorted(MODELS)}.")
selected = MODELS[model_choice]

# Set this when the notebook's own role is not a SageMaker execution role.
execution_role = os.environ.get("SAGEMAKER_EXECUTION_ROLE_ARN")
bucket = os.environ.get("SAGEMAKER_ARTIFACT_BUCKET", f"sagemaker-{region}-{account_id}")
# A CPU instance. 8 vCPUs and 16 GiB are comfortable for the ~3 GiB FP32 worker;
# try ml.c7i.xlarge or ml.m7i.xlarge (4 vCPUs) to find the cheapest viable size.
instance_type = os.environ.get("SAGEMAKER_INSTANCE_TYPE", "ml.c7i.2xlarge")
# Torch intra-op threads in the worker; empty uses every vCPU of the instance.
cpu_threads = os.environ.get("SAGEMAKER_CPU_THREADS", "")
# Set this to reuse a serving image you already built; otherwise section 2 builds one.
inference_image = os.environ.get("SAGEMAKER_INFERENCE_IMAGE_URI")
codebuild_role_arn = os.environ.get("CODEBUILD_ROLE_ARN")
cpu_base_image = os.environ.get(
    "SAGEMAKER_SERVING_BASE_IMAGE",
    "public.ecr.aws/deep-learning-containers/pytorch:2.14-cpu-amzn2023-sagemaker",
)
name = f"decision-{model_choice}-cpu-{uuid.uuid4().hex[:12]}"
artifact_key = f"decision-{model_choice}-cpu/{name}/model.tar.gz"
artifact_version = None
example_dir = Path.cwd()
archive_path = example_dir / f"{name}.tar.gz"
for filename in ["inference.py", "requirements.txt", "example-request.json"]:
    if not (example_dir / filename).is_file():
        raise FileNotFoundError(f"Run the notebook from its example directory: missing {filename}")
print(f"Region: {region}; model: {selected['model_id']}; instance: {instance_type}; endpoint: {name}")

### Resolve the execution role

In Studio the current role may already be appropriate. Outside Studio,
set `SAGEMAKER_EXECUTION_ROLE_ARN` to an existing SageMaker role. This cell
reads the role's actual ARN through IAM, preserving paths such as
`service-role/`, and checks its SageMaker trust.

In [ ]:
if execution_role is None:
    caller_arn = identity["Arn"]
    if ":assumed-role/" not in caller_arn:
        raise ValueError("Set SAGEMAKER_EXECUTION_ROLE_ARN to an existing SageMaker execution role.")
    role_name = caller_arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
else:
    role_name = execution_role.rsplit("/", 1)[-1]
role_info = iam.get_role(RoleName=role_name)["Role"]
if execution_role is not None and execution_role != role_info["Arn"]:
    raise ValueError("The configured execution-role ARN does not match the resolved IAM role.")
execution_role = role_info["Arn"]
trusts_sagemaker = any(
    statement.get("Effect") == "Allow"
    and "sagemaker.amazonaws.com" in (
        [statement.get("Principal", {}).get("Service")]
        if isinstance(statement.get("Principal", {}).get("Service"), str)
        else statement.get("Principal", {}).get("Service", [])
    )
    for statement in role_info["AssumeRolePolicyDocument"]["Statement"]
)
if not trusts_sagemaker:
    raise ValueError(
        "The current role does not trust SageMaker. Set SAGEMAKER_EXECUTION_ROLE_ARN "
        "to an appropriate existing execution role and rerun setup."
    )
print("Execution role resolved and SageMaker trust verified.")

## 2. Build the CPU serving image

There is no AWS PyTorch inference DLC after 2.6, and that image is past its end of
support. This example runs on the maintained PyTorch 2.14 **CPU** SageMaker DLC through
the small shared image in
[03-features/decision-model-serving-image](../../../03-features/decision-model-serving-image/),
built with its `BASE_IMAGE` set to the CPU DLC. It adds `serve`, which answers `/ping`
and `/invocations` by calling this example's `model_fn` and `transform_fn`.

AWS CodeBuild builds the image and pushes it to a private Amazon ECR repository in your
account. Set `CODEBUILD_ROLE_ARN` to a CodeBuild service role that can read the artifact
bucket, write build logs, and push to Amazon ECR. To reuse an image you already built,
set `SAGEMAKER_INFERENCE_IMAGE_URI` and this cell skips the build.

In [ ]:
import sys

if inference_image is None:
    if not codebuild_role_arn:
        raise ValueError("Set CODEBUILD_ROLE_ARN, or SAGEMAKER_INFERENCE_IMAGE_URI to reuse an image.")
    image_dir = example_dir.parents[2] / "03-features" / "decision-model-serving-image"
    if not image_dir.is_dir():
        raise FileNotFoundError(f"{image_dir} not found. Run this notebook from a clone of the repository.")
    sys.path.insert(0, str(image_dir))
    from build_image import build_image

    inference_image = build_image(
        region=region, bucket=bucket, codebuild_role_arn=codebuild_role_arn,
        base_image=cpu_base_image, tag_prefix="pt214-cpu",
    )
print(f"Serving image: {inference_image}")

## 3. Prepare the model artifact

Package the companion handler and requirements under `code/`, as expected by
the serving image. The requirements file installs nothing: PyTorch, Transformers,
and the Hugging Face libraries come from the image, and the Decision 2.0 runtime
ships inside the pinned model repository. The worker loads the pinned Hugging Face
checkpoint and its bundled runtime using `trust_remote_code=True`. Review the
published code before adopting it for another workload.

The default artifact bucket is shared and is preserved during cleanup. Capture
the uploaded object's version directly from the upload response.

In [ ]:
try:
    s3.head_bucket(Bucket=bucket)
except ClientError as error:
    if error.response["Error"]["Code"] not in {"404", "NoSuchBucket", "NotFound"}:
        raise
    create_args = {"Bucket": bucket}
    if region != "us-east-1":
        create_args["CreateBucketConfiguration"] = {"LocationConstraint": region}
    s3.create_bucket(**create_args)
    s3.put_public_access_block(
        Bucket=bucket,
        PublicAccessBlockConfiguration={
            "BlockPublicAcls": True, "IgnorePublicAcls": True,
            "BlockPublicPolicy": True, "RestrictPublicBuckets": True,
        },
    )

with tarfile.open(archive_path, "w:gz") as archive:
    for filename in ["inference.py", "requirements.txt"]:
        archive.add(example_dir / filename, arcname=f"code/{filename}")
with archive_path.open("rb") as artifact:
    uploaded = s3.put_object(Bucket=bucket, Key=artifact_key, Body=artifact)
artifact_version = uploaded.get("VersionId")
model_data_url = f"s3://{bucket}/{artifact_key}"
print("Inference artifact uploaded.")

## 4. Create the model, endpoint configuration, and endpoint

SageMaker allows 1,800 seconds for the container to pass its health check, which
covers pulling the image and downloading the 1.4 GB checkpoint from Hugging Face.
The serving image runs one process, so the weights load once, and the adapter
serializes inference. The model's native runtime fixes CPU numerics at FP32; do
not override its dtype.

`DECISION_MODEL_ID` and `DECISION_MODEL_REVISION` select the pinned checkpoint;
`DECISION_THREADS` sets Torch intra-op threads (default: every vCPU). No
`InferenceAmiVersion` or `VolumeSizeInGB` is set for a CPU instance.

In [ ]:
environment = {
    "SAGEMAKER_PROGRAM": "inference.py",
    "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
    "DECISION_MODEL_ID": selected["model_id"],
    "DECISION_MODEL_REVISION": selected["revision"],
    "HF_HOME": "/tmp/huggingface",
    "HF_HUB_DISABLE_TELEMETRY": "1",
    "TOKENIZERS_PARALLELISM": "false",
}
if cpu_threads:
    environment["DECISION_THREADS"] = cpu_threads
sm.create_model(
    ModelName=name,
    ExecutionRoleArn=execution_role,
    PrimaryContainer={
        "Image": inference_image,
        "ModelDataUrl": model_data_url,
        "Environment": environment,
    },
)
sm.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic", "ModelName": name,
        "InitialInstanceCount": 1, "InstanceType": instance_type,
        "InitialVariantWeight": 1.0,
        "ModelDataDownloadTimeoutInSeconds": 900,
        "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
    }],
)
sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
print(f"Endpoint {name} is deploying.")

In [ ]:
sm.get_waiter("endpoint_in_service").wait(
    EndpointName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 120}
)
endpoint = sm.describe_endpoint(EndpointName=name)
assert endpoint["EndpointStatus"] == "InService"
assert endpoint["EndpointConfigName"] == name
assert sum(v["CurrentInstanceCount"] for v in endpoint["ProductionVariants"]) == 1
print("Endpoint is InService with one instance.")

## 5. Verify the runtime and invoke typed questions

The adapter accepts `{"state": ..., "questions": {...}}` through SageMaker's
IAM-authenticated API and calls `model.system_one`. Its diagnostics operation
reports effective versions, the CPU model and thread settings, checkpoint
metadata, and the worker's peak resident memory.

In [ ]:
def invoke(payload):
    response = runtime.invoke_endpoint(
        EndpointName=name, ContentType="application/json",
        Accept="application/json", Body=json.dumps(payload),
    )
    return json.loads(response["Body"].read())

diagnostics = invoke({"operation": "diagnostics"})
assert diagnostics["device"] == "cpu"
assert diagnostics["cuda_available"] is False
assert diagnostics["model_revision"] == selected["revision"]
assert diagnostics["model"] == selected["model_name"]
assert diagnostics["loaded_parameters"] == selected["loaded_parameters"]
assert diagnostics["max_input_tokens"] == selected["max_input_tokens"]
assert diagnostics["parameter_dtypes"] == {"torch.float32": selected["loaded_parameters"]}
assert diagnostics["versions"]["torch"].startswith("2.14.")
assert diagnostics["versions"]["transformers"] == "5.17.0"
assert diagnostics["torch_threads"] == int(cpu_threads or diagnostics["cpu_count"])
print(json.dumps(diagnostics, indent=2))

### Support-ticket routing: Choice, Yes/No, and Score

A damaged order arrives with a receipt and a same-day replacement request. Ask
which team should handle it, whether the customer has a receipt, and how urgent
it is. One comparative `choice` with concrete candidate descriptions is the
recommended shape; the harness owns thresholds and what happens next.

In [ ]:
request = json.loads((example_dir / "example-request.json").read_text())
result = invoke(request)
print(json.dumps(result, indent=2))

In [ ]:
assert result["model"] == diagnostics["model"]
answers = result["answers"]
assert set(answers) == {"route", "receipt", "urgency"}
assert answers["receipt"]["type"] == "noul"
assert math.isfinite(answers["receipt"]["noul"])
assert 0 <= answers["receipt"]["noul"] <= 1
for key, kind in [("route", "choice"), ("urgency", "score")]:
    answer = answers[key]
    assert answer["type"] == kind
    probabilities = list(answer["probabilities"].values())
    assert all(math.isfinite(p) and 0 <= p <= 1 for p in probabilities)
    assert abs(sum(probabilities) - 1) <= 0.001
assert answers["route"]["choice"] in request["questions"]["route"]["criteria"]
assert math.isfinite(answers["urgency"]["score"])
assert 0 <= answers["urgency"]["score"] <= 2
print("All three answer types verified.")

### Basic expected-answer smoke checks

These simple cases check the serving path. They are not a representative
benchmark of decision quality or calibration.

In [ ]:
assert answers["route"]["choice"] == "returns"

city_question = {"city_supplied": {
    "type": "noul", "instructions": "Did the user explicitly provide a city?",
}}
explicit_city = invoke({
    "state": "The user says: What is the current weather in Paris?",
    "questions": city_question,
})
missing_city = invoke({
    "state": "The user says: What is the current weather?",
    "questions": city_question,
})
assert explicit_city["answers"]["city_supplied"]["noul"] >= 0.5
assert missing_city["answers"]["city_supplied"]["noul"] < 0.5
print("Basic expected-answer checks: 3/3.")

### Repeatability and server latency

Three repeats give a hosting smoke measurement, not a production latency
estimate. `latency_ms` measures server-side inference; SDK/network time is
measured separately in the following scaling check.

In [ ]:
repeats = [invoke(request) for _ in range(3)]
assert all(item["answers"] == result["answers"] for item in repeats)
print(f"Median warm server latency: {statistics.median(item['latency_ms'] for item in repeats):.2f} ms")

### Reject oversized state instead of silently truncating it

The oversized request has valid schema and exceeds the checkpoint's token
budget (8,192 for Kai, 16,384 for Eos). The adapter turns native context-window
errors into HTTP 422; SageMaker wraps this in `ModelError` and may omit the
error body.

In [ ]:
oversized_request = {
    "state": "weather city location " * 10000,
    "questions": city_question,
}
try:
    invoke(oversized_request)
except runtime.exceptions.ModelError as error:
    assert error.response.get("OriginalStatusCode") == 422, error.response
    print("Oversized state rejected with model HTTP 422.")
else:
    raise AssertionError("The strict context window did not reject oversized state.")

### Bounded multi-question scaling

Run 1, 8, 32, and 64 independent question IDs over the same short state. These
are repeatability and hosting checks, not a decision-quality benchmark. The
default exact runtime batches questions and repeats their state in each
sequence; on CPU every request runs as one FP32 forward batch. Shared-context
reuse is disabled by default and is not enabled in this example.

Measure three requests per question count. Server latency excludes SDK/network
time; round-trip latency includes it. Worker RSS is the lifetime high-water
mark, including startup and every preceding request, and does not include every
process on the hosting instance. These results apply only to this short input,
runtime, thread setting, and instance. `InvokeEndpoint` allows 60 seconds per
request; on a smaller instance, drop the 64-question step if it approaches that.

In [ ]:
scaling_results = []
for count in [1, 8, 32, 64]:
    payload = {
        "state": "The user explicitly requested the current weather in Paris.",
        "questions": {
            f"city_{index}": {
                "type": "noul",
                "instructions": "Did the user explicitly provide a city?",
            }
            for index in range(count)
        },
    }
    server_times, round_trip_times = [], []
    baseline_answers = None
    for _ in range(3):
        started = time.perf_counter()
        response = invoke(payload)
        round_trip_times.append((time.perf_counter() - started) * 1000)
        assert set(response["answers"]) == set(payload["questions"])
        for answer in response["answers"].values():
            assert answer["type"] == "noul" and math.isfinite(answer["noul"])
            assert 0 <= answer["noul"] <= 1
        if baseline_answers is None:
            baseline_answers = response["answers"]
        else:
            assert response["answers"] == baseline_answers
        server_times.append(response["latency_ms"])
    memory = invoke({"operation": "diagnostics"})
    scaling_results.append({
        "questions": count,
        "server_median_ms": round(statistics.median(server_times), 2),
        "round_trip_median_ms": round(statistics.median(round_trip_times), 2),
        "worker_peak_rss_bytes": memory["worker_peak_rss_bytes"],
        "worker_peak_rss_scope": "worker_lifetime",
    })
print(json.dumps({
    "instance_type": instance_type,
    "processor": memory["processor"],
    "torch_threads": memory["torch_threads"],
    "results": scaling_results,
}, indent=2))

## 6. Cleanup

Delete this example's endpoint and wait before deleting its configuration and
model. Remove every S3 version and delete marker for this example's unique
artifact key, including repeated uploads. Preserve the shared bucket and role.
Attempt independent cleanup even if another operation fails; report failures.

The serving image, its Amazon ECR repository (`decision-model-serving`), and the
CodeBuild project are kept so later runs can reuse them; see the README to
remove them. Late endpoint log delivery can recreate the CloudWatch log group
after deletion. Rerun cleanup after delivery settles if the group reappears.

In [ ]:
def ignore_missing(operation, **arguments):
    try:
        return operation(**arguments)
    except ClientError as error:
        message = error.response["Error"].get("Message", "")
        error_code = error.response["Error"]["Code"]
        if error_code == "ResourceNotFoundException" or (
            error_code == "ValidationException"
            and ("Could not find" in message or "does not exist" in message)
        ):
            return None
        raise

def delete_endpoint():
    if ignore_missing(sm.describe_endpoint, EndpointName=name) is not None:
        sm.delete_endpoint(EndpointName=name)
        sm.get_waiter("endpoint_deleted").wait(
            EndpointName=name, WaiterConfig={"Delay": 15, "MaxAttempts": 120}
        )

def artifact_versions():
    return [
        {"Key": item["Key"], "VersionId": item["VersionId"]}
        for page in s3.get_paginator("list_object_versions").paginate(
            Bucket=bucket, Prefix=artifact_key
        )
        for group in ("Versions", "DeleteMarkers")
        for item in page.get(group, [])
        if item["Key"] == artifact_key
    ]

def delete_artifact():
    for item in artifact_versions():
        s3.delete_object(Bucket=bucket, **item)
    if artifact_versions():
        raise RuntimeError("S3 artifact versions remain after cleanup.")

cleanup_errors = []
def attempt_cleanup(label, operation, *positional, **arguments):
    try:
        operation(*positional, **arguments)
        return True
    except Exception as error:
        cleanup_errors.append(f"{label}: {error}")
        return False

try:
    if attempt_cleanup("endpoint", delete_endpoint):
        attempt_cleanup(
            "endpoint configuration", ignore_missing,
            sm.delete_endpoint_config, EndpointConfigName=name,
        )
        attempt_cleanup("model", ignore_missing, sm.delete_model, ModelName=name)
    attempt_cleanup("S3 artifact", delete_artifact)
    attempt_cleanup(
        "endpoint logs", ignore_missing,
        logs.delete_log_group, logGroupName=f"/aws/sagemaker/Endpoints/{name}",
    )
finally:
    attempt_cleanup("local archive", archive_path.unlink, missing_ok=True)

if cleanup_errors:
    raise RuntimeError("Cleanup incomplete: " + "; ".join(cleanup_errors))
assert ignore_missing(sm.describe_endpoint, EndpointName=name) is None
assert ignore_missing(sm.describe_endpoint_config, EndpointConfigName=name) is None
assert ignore_missing(sm.describe_model, ModelName=name) is None
print("Endpoint, configuration, model, all artifact versions, and endpoint logs cleaned up.")